In [ ]:
# =============================================================
# PART 1 — Imports, Dataset, Model, Loss
# =============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image

import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import numpy as np
import imageio.v2 as imageio
import os
from scipy import linalg

# =============================================================
# 4.2 DATASET PREPARATION
# =============================================================

batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor()   # Keeps it in [0,1] for BCE loss
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform)

test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

os.makedirs("outputs", exist_ok=True)
os.makedirs("outputs/recon_latent_frames", exist_ok=True)
os.makedirs("outputs/beta_latent", exist_ok=True)
os.makedirs("outputs/frozen", exist_ok=True)

# =============================================================
# 4.3 MODEL ARCHITECTURE
# =============================================================

class Encoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h = self.net(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28*28),
            nn.Sigmoid()
        )

    def forward(self, z):
        img = self.net(z)
        return img.view(-1, 1, 28, 28)


class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar, z


# =============================================================
# 4.4 LOSS FUNCTION
# =============================================================

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    BCE = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')

    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    total = BCE + beta * KLD
    return total, BCE, KLD

# =============================================================
# FUNCTION — LATENT SPACE RECON (GIF FRAMES)
# =============================================================

def plot_images_in_latent_space(model, imgs, frame_id, epoch, batch):
    """Places reconstructed thumbnails at latent (z1,z2) coordinates with black background."""
    model.eval()
    with torch.no_grad():
        recon, mu, logvar, z = model(imgs)

    mu = mu.numpy()
    recon = recon.numpy()

    fig = plt.figure(figsize=(8, 8), dpi=150)
    ax = fig.add_subplot(111)

    # black background
    ax.set_facecolor("black")

    # fixed bounds
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)

    # fixed margins
    plt.subplots_adjust(left=0.05, right=0.95, top=0.92, bottom=0.05)

    # text overlay like your reference video
    ax.text(-3.8, 3.6, f"Epoch: {epoch}   Batch: {batch}",
            fontsize=12, color="white", weight="bold")

    # reconstruct up to 80 images
    max_imgs = min(len(mu), 80)

    for i in range(max_imgs):
        img = recon[i].squeeze()
        imagebox = OffsetImage(img, cmap="gray", zoom=0.9)
        ab = AnnotationBbox(imagebox, (mu[i, 0], mu[i, 1]), frameon=False)
        ax.add_artist(ab)

    ax.set_xticks([])
    ax.set_yticks([])

    # save frame
    plt.savefig(f"outputs/recon_latent_frames/frame_{frame_id}.png")
    plt.close(fig)


# =============================================================
# TRAINING (4.5) + LATENT GIF GENERATION (4.6 requirement)
# =============================================================

latent_dim = 2
model = VAE(latent_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15
loss_history = []
frame_id = 0

print("\nTraining VAE and generating latent reconstruction GIF...\n")

for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    for batch_idx, (imgs, _) in enumerate(train_loader):

        # forward pass
        recon, mu, logvar, z = model(imgs)
        loss, BCE, KLD = vae_loss(recon, imgs, mu, logvar)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # save GIF frame every 40 batches
        if batch_idx % 40 == 0:
            plot_images_in_latent_space(model, imgs[:80], frame_id, epoch, batch_idx)
            frame_id += 1

    avg_loss = total_loss / len(train_dataset)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch}/{EPOCHS}  Loss = {avg_loss:.4f}")


# =============================================================
# LOSS PLOT
# =============================================================

plt.plot(loss_history)
plt.title("VAE Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.savefig("outputs/loss_curve.png")
plt.close()


# =============================================================
# CREATE LATENT EVOLUTION GIF
# =============================================================

frames = []
for i in range(frame_id):
    img = imageio.imread(f"outputs/recon_latent_frames/frame_{i}.png")
    frames.append(img)

imageio.mimsave("outputs/reconstruction_latent_evolution.gif",
                frames, duration=0.30)

print("\nGIF saved at: outputs/reconstruction_latent_evolution.gif\n")

# =============================================================
# 4.6 β-VAE EXPERIMENTAL ANALYSIS
# =============================================================

# Choose 3 classes for color-coded latent visualization
chosen_classes = [0, 2, 5]     # (T-shirt/top, Pullover, Sandal)
class_colors = {0: "red", 2: "green", 5: "blue"}


def filter_classes(loader, classes):
    """Return a dataloader containing ONLY the chosen classes."""
    imgs_filtered = []
    labels_filtered = []

    for imgs, labels in loader:
        mask = torch.zeros_like(labels, dtype=torch.bool)
        for c in classes:
            mask |= (labels == c)
        imgs_filtered.append(imgs[mask])
        labels_filtered.append(labels[mask])

    X = torch.cat(imgs_filtered)
    Y = torch.cat(labels_filtered)

    filtered = torch.utils.data.TensorDataset(X, Y)
    return DataLoader(filtered, batch_size=256, shuffle=False)


filtered_test_loader = filter_classes(test_loader, chosen_classes)


# -------------------------------------------------------------
# TRAIN β-VAE
# -------------------------------------------------------------

def train_beta_vae(beta):
    """Trains a new VAE with the given β."""
    vae = VAE(latent_dim=2)
    optim_beta = optim.Adam(vae.parameters(), lr=1e-3)

    for epoch in range(6):  # small number of epochs—enough for visualization
        for imgs, _ in train_loader:
            recon, mu, logvar, z = vae(imgs)
            loss, bce, kld = vae_loss(recon, imgs, mu, logvar, beta)

            optim_beta.zero_grad()
            loss.backward()
            optim_beta.step()

    return vae


# -------------------------------------------------------------
# LATENT SPACE PLOT (COLOR CODED)
# -------------------------------------------------------------

def plot_beta_latent(model, beta_value):
    Z = []
    Y = []

    with torch.no_grad():
        for imgs, labels in filtered_test_loader:
            recon, mu, logvar, z = model(imgs)
            Z.append(mu.numpy())
            Y.append(labels.numpy())

    Z = np.concatenate(Z)
    Y = np.concatenate(Y)

    plt.figure(figsize=(7, 7))
    plt.title(f"β-VAE Latent Space (β={beta_value})")

    for cls in chosen_classes:
        idx = (Y == cls)
        plt.scatter(Z[idx, 0], Z[idx, 1],
                    c=class_colors[cls], s=8, label=f"Class {cls}")

    plt.legend()
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)
    plt.grid(True)
    plt.savefig(f"outputs/beta_latent/beta_{beta_value}.png")
    plt.close()


# -------------------------------------------------------------
# RUN EXPERIMENT FOR β = 0.1, 0.5, 1
# -------------------------------------------------------------

beta_values = [0.1, 0.5, 1]
beta_models = {}

for b in beta_values:
    print(f"\nTraining β-VAE for β={b} ...\n")
    model_b = train_beta_vae(b)
    beta_models[b] = model_b
    plot_beta_latent(model_b, b)


# -------------------------------------------------------------
# CREATE GIF OF β-VAE LATENT SPACES
# -------------------------------------------------------------

beta_frames = []

for b in beta_values:
    img = imageio.imread(f"outputs/beta_latent/beta_{b}.png")
    beta_frames.append(img)

imageio.mimsave("outputs/beta_latent/beta_latent_evolution.gif",
                beta_frames, duration=1.0)

print("\nβ-VAE latent GIF saved at: outputs/beta_latent/beta_latent_evolution.gif\n")


# -------------------------------------------------------------
# METRIC TABLE FOR β-VAE (qualitative + quantitative)
# -------------------------------------------------------------

beta_table = []

for b in beta_values:
    model_b = beta_models[b]

    total_bce = 0
    total_kld = 0
    count = 0

    with torch.no_grad():
        for imgs, _ in test_loader:
            recon, mu, logvar, z = model_b(imgs)
            _, bce, kld = vae_loss(recon, imgs, mu, logvar, beta=1)
            total_bce += bce.item()
            total_kld += kld.item()
            count += imgs.size(0)

    beta_table.append([
        b,
        total_bce / count,
        total_kld / count
    ])

# save metric table
import pandas as pd
df = pd.DataFrame(beta_table, columns=["beta", "avg_bce", "avg_kld"])
df.to_csv("outputs/beta_latent/beta_metrics.csv", index=False)

print("β-VAE metrics saved to outputs/beta_latent/beta_metrics.csv\n")

# =============================================================
# 4.7 EVALUATION & VISUALIZATION
# =============================================================

# ---- (A) Original vs Reconstructed images ----
print("\nSaving original vs reconstructed images...")

model.eval()
imgs, _ = next(iter(test_loader))

with torch.no_grad():
    recon, mu, logvar, z = model(imgs)

comparison = torch.cat([imgs[:8], recon[:8]], dim=0)
save_image(comparison, "outputs/reconstruction.png")
print("Saved: outputs/reconstruction.png")


# ---- (B) Sampling new images from N(0,I) ----
print("\nGenerating random samples from N(0,I)...")

z_random = torch.randn(16, latent_dim)
samples = model.decoder(z_random)
save_image(samples, "outputs/random_samples.png")

print("Saved: outputs/random_samples.png")


# =============================================================
# (C) Compute FID Score for generated images
# =============================================================

def compute_fid(real_images, fake_images):
    """
    Computes FID using activations from a simple flattened feature space.
    NOTE: This is a simplified FID appropriate for academic assignments.
    """

    # Flatten images
    real = real_images.reshape(real_images.shape[0], -1)
    fake = fake_images.reshape(fake_images.shape[0], -1)

    # Means & covariance
    mu1 = real.mean(axis=0)
    mu2 = fake.mean(axis=0)

    sigma1 = np.cov(real, rowvar=False)
    sigma2 = np.cov(fake, rowvar=False)

    # FID computation
    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1 @ sigma2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(fid)


# Create real and fake datasets for FID
real_imgs = imgs[:200].numpy()
fake_imgs = model.decoder(torch.randn(200, latent_dim)).detach().numpy()

fid_score = compute_fid(real_imgs, fake_imgs)

# Specify UTF-8 encoding when writing text files to avoid UnicodeEncodeError on some platforms
with open("outputs/FID_score.txt", "w", encoding="utf-8") as f:
    f.write(f"FID Score = {fid_score:.4f}")

print(f"\nFID Score computed and saved: {fid_score:.4f}")
print("Saved: outputs/FID_score.txt")


# =============================================================
# 4.8 FROZEN LATENT PARAMETER EXPERIMENT
# =============================================================

print("\nRunning frozen latent parameter experiments...")

def sample_frozen(mu_value, sigma_value, n=16):
    """μ fixed to 0, σ fixed to constant value."""
    z = mu_value + sigma_value * torch.randn(n, latent_dim)
    return model.decoder(z)

# ----- Generate images for σ = 0.1, 0.5, 1 -----
sigma_values = [0.1, 0.5, 1.0]

for s in sigma_values:
    imgs = sample_frozen(0, s)
    save_image(imgs, f"outputs/frozen/frozen_sigma_{s}.png")
    print(f"Saved: outputs/frozen/frozen_sigma_{s}.png")

# Also save stochastic (μ, σ learned)
z_std = model.reparameterize(mu[:16], logvar[:16])
stochastic_samples = model.decoder(z_std)
save_image(stochastic_samples, "outputs/frozen/stochastic_samples.png")

print("Saved: outputs/frozen/stochastic_samples.png")



Training VAE and generating latent reconstruction GIF...

Epoch 1/15  Loss = 288.2769
Epoch 2/15  Loss = 266.9135
Epoch 3/15  Loss = 263.7177
Epoch 4/15  Loss = 261.8998
Epoch 5/15  Loss = 260.6267
Epoch 6/15  Loss = 259.7468
Epoch 7/15  Loss = 259.0282
Epoch 8/15  Loss = 258.4292
Epoch 9/15  Loss = 257.9304
Epoch 10/15  Loss = 257.7598
Epoch 11/15  Loss = 257.1543
Epoch 12/15  Loss = 256.6532
Epoch 13/15  Loss = 256.2528
Epoch 14/15  Loss = 255.9582
Epoch 15/15  Loss = 255.7826

GIF saved at: outputs/reconstruction_latent_evolution.gif


Training β-VAE for β=0.1 ...


Training β-VAE for β=0.5 ...


Training β-VAE for β=1 ...


β-VAE latent GIF saved at: outputs/beta_latent/beta_latent_evolution.gif

β-VAE metrics saved to outputs/beta_latent/beta_metrics.csv


Saving original vs reconstructed images...
Saved: outputs/reconstruction.png

Generating random samples from N(0,I)...
Saved: outputs/random_samples.png

FID Score computed and saved: 18.6063
Saved: outputs/FID_score.txt

Runni

: 